 # Gemini API Quota Tester

 This notebook tests your Gemini API key directly against the Google Generative AI endpoints to diagnose quota/429 issues.

In [3]:
# Install dependencies if needed
!pip install requests python-dotenv


In [4]:
import os
import requests
import json
from dotenv import load_dotenv

# 1. Load the API key from your .env file
# This assumes the .env file is in the same directory as this notebook
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("❌ Error: GEMINI_API_KEY not found in .env file.")
else:
    print(f"✅ Found API Key: {api_key[:8]}...{api_key[-4:]}")


✅ Found API Key: AIzaSyAZ...qynU


In [ ]:
curl "https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent" \
  -H "x-goog-api-key: $GEMINI_API_KEY" \
  -H 'Content-Type: application/json' \
  -X POST \
  -d '{
    "contents": [
      {
        "parts": [
          {
            "text": "Explain how AI works in a few words"
          }
        ]
      }
    ]
  }'

In [ ]:
# 2. Define the test function
def test_gemini_model(model_name="gemini-1.5-flash-8b", api_version="v1beta"):
    url = f"https://generativelanguage.googleapis.com/{api_version}/models/{model_name}:generateContent"
    
    headers = {
        f"x-goog-api-key": api_key,
        "Content-Type": "application/json"
    }
    
    payload = {
        "contents": [{
            "parts": [{
                "text": "Say 'Connection Successful' if you can read this."
            }]
        }]
    }
    
    print(f"\n--- Testing Model: {model_name} ({api_version}) ---")
    try:
        response = requests.post(url, headers=headers, json=payload)
        
        if response.status_code == 200:
            result = response.json()
            text = result['candidates'][0]['content']['parts'][0]['text']
            print(f"🟢 SUCCESS! Response: {text.strip()}")
        else:
            print(f"🔴 FAILED (Status {response.status_code})")
            print(f"Error Detail: {response.text}")
            
    except Exception as e:
        print(f"💥 Exception: {str(e)}")


In [6]:
# 3. Run the tests
# We test a few variants to see which one (if any) works for your key
if api_key:
    # Test 1: The standard v1beta 1.5-flash
    test_gemini_model("gemini-1.5-flash", "v1beta")
    
    # Test 2: The 8B version (often has more free quota)
    test_gemini_model("gemini-1.5-flash-8b", "v1beta")
    
    # Test 3: The v1 (Production) endpoint
    test_gemini_model("gemini-1.5-flash", "v1")
    
    # Test 4: 2.0 Flash Lite (the one we just tried)
    test_gemini_model("gemini-2.0-flash-lite", "v1beta")



--- Testing Model: gemini-1.5-flash (v1beta) ---
🔴 FAILED (Status 404)
Error Detail: {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}


--- Testing Model: gemini-1.5-flash-8b (v1beta) ---
🔴 FAILED (Status 404)
Error Detail: {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash-8b is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}


--- Testing Model: gemini-1.5-flash (v1) ---
🔴 FAILED (Status 404)
Error Detail: {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1, or is not supported for generateContent. Call ListModels to see the list of available models and their 

In [11]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI identifies **patterns in data** to make **predictions** and solve problems.


TypeError: 'Tokens' object is not callable

In [10]:
!pip install google-genai

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 35.4 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 39.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.36.0
    Uninstalling google-auth-2.36.0:
      Successfully uninstalled google-auth-2.36.0
